# 4장. pandas로 데이터에 질문하기

이 노트북은 데이터를 선택·필터링·정렬하고, 안전하게 병합한 뒤 완료 주문 기준 요약표를 만드는 실습 자료입니다.


## 0. 제출 정보
- 이름: 유은송 
- GitHub ID: song-03
- 작성일: 2026.09.19
- 최종 제출 URL: https://github.com/song-03/llm-data-analysis-study/blob/main/chapter04/chapter04.ipynb

## 1. 질문과 필요한 데이터 선택
### 내가 확인하려는 질문
completed 주문 기준 어떤 상품 카테고리의 금액이 가장 큰가?
### 사용한 파일/컬럼

- `orders.csv`: `order_id`, `order_status`
- `order_items.csv`: `order_id`, `product_id`, `quantity`, `unit_price`
- `products.csv`: `product_id`, `category`

### 결과 관찰
분석질문은 `completed 주문 기준 어떤 상품 카테고리의 금액이 가장 큰가?` 로 설정했다.
필요한 데이터는 `orders.csv`,`order_items.csv, `products.csv`이며, 금액의 경우 quantity * unit_price로 계산한 뒤 category 단위로 집계하여 계산하기로 하였다.

### 나의 해석과 판단
우선 주문 금액을 계산해야 하므로 이를 위한 파일와 컬럼을 선택해야 한다. 
먼저 order_items에서 quantity * unit_price로 주문 상세별 금액인 line_total을 계산해야 하므로 두 데이터가 필요하다. 이후 completed 상태인 주문만 골라야 하는데, 이를 위해서 order_id 기준으로 orders를 연결해 order_status가 completed인 주문만 골라내고, product_id로 products를 연결해서 category를 가져와야 한다. 이를 통해서카테고리별 line_total 합계를 비교해 원하는 결과를 얻어야 하기 때문에 이러한 데이터들을 선택했다.
상태의 경우 completed만 선택했는데, refunded나 cancelled는 내가 확인하려는 질문과 관계가 없기 때문에 제외했다. 이후 이를 매출 분석 혹은 이를 이용한 마케팅 전략 수립 등에 사용해야 하는데, 환불이나 취소된 건은 실제 주문으로 이어진 건이 아니기 때문에 제외했다. 환불, 취소된 건은 추후에 따로 환불이 많이 되는 상품, 취소가 많이 되는 상품 등 다른 질문을 세워 분석할 수 있을 것이다.

### 업무·분석적 의미
질문을 설정할 때 필요한 범위로 확실히 좁히면 (ex. 이번 경우 completed 주문) 분석 범위와 필요한 데이터를 명확하게 확인할 수 있다. 고객의 성별, 나이 등 속성처럼 필요하지 않은 정보는 제외할 수 있고, 취소나 환불된 주문이 결과에 섞이는 것도 방지할 수 있다. 필요없는 정보는 제외해 데이터 과부하 위험도 줄일 수 있다.

### 한계와 추가 확인 사항
현재는 분석 질문, 계산 방법 등만 정의했고 아직 결과를 확인하지는 못했다. 또한 이후 실제 데이터를 확인해야 한다. 데이터에서 completed된 주문에서 내가 필요로 하는 데이터가 이상값 없이 모두 되는지, line_total 값이 실제로 정상적으로 계산되는지, id 값들이 고유한 것이 맞는지, 매칭되지 않는 데이터는 없는지 등 확인해야 한다. 


### 1. 프로젝트 루트와 실행 환경 확인

VS Code에서 Notebook을 실행하면 현재 작업 폴더가 프로젝트 루트 또는 `notebooks` 폴더일 수 있습니다. 아래 코드는 상위 폴더를 확인해 프로젝트 루트를 찾습니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Python 실행 파일:', sys.executable)
print('현재 작업 폴더:', Path.cwd())
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)


Python 실행 파일: c:\dev\llm-data-analysis-course\.venv\Scripts\python.exe
현재 작업 폴더: c:\dev\llm-data-analysis-course\notebooks
프로젝트 루트: C:\dev\llm-data-analysis-course
데이터 폴더: C:\dev\llm-data-analysis-course\data\raw
결과 저장 폴더: C:\dev\llm-data-analysis-course\reports


### 2. 데이터 파일 확인과 불러오기

파일이 없다면 프로젝트 루트에서 `python scripts/generate_sample_data.py`를 먼저 실행하세요.


In [2]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')


데이터 불러오기 완료


### 3. 데이터 구조와 필수 컬럼 확인

LLM이 작성한 코드가 실제 컬럼명과 일치하는지 먼저 확인합니다.


In [3]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')


customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


In [18]:
print(all(c in orders.columns for c in ["order_id", "order_status"]))
print(all(c in order_items.columns for c in ["order_id", "product_id", "quantity", "unit_price"]))
print(all(c in products.columns for c in ["product_id", "category"]))

True
True
True


### 4. 컬럼 선택과 행 필터링

이 저장소의 샘플 데이터는 도시명을 `서울`, `부산`처럼 한글로 생성합니다. 필터링 전에 실제 값을 확인합니다.


In [4]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customer_basic.head())

print('도시별 고객 수')
display(customers['city'].value_counts())

customers_over_30 = customers[customers['age'] >= 30]
seoul_customers = customers[customers['city'] == '서울']
city_customers = customers[customers['city'].isin(['서울', '부산'])]

print('30세 이상 고객 수:', len(customers_over_30))
print('서울 고객 수:', len(seoul_customers))
print('서울 또는 부산 고객 수:', len(city_customers))


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


도시별 고객 수


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

30세 이상 고객 수: 111
서울 고객 수: 15
서울 또는 부산 고객 수: 31


### 5. 정렬과 주문 상태 확인


In [5]:
display(products.sort_values('price', ascending=False).head(10))
display(customers.sort_values('age', ascending=False).head(10))

print('주문 상태별 건수')
display(orders['order_status'].value_counts())


,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,customer_id,name,gender,age,city,signup_date
14,15,장정식,M,69,서울,2026-06-04
8,9,송지민,M,69,서울,2025-10-19
54,55,윤영철,M,69,광주,2023-07-31
78,79,이서준,M,69,부산,2023-07-14
136,137,김선영,M,68,울산,2024-11-28
45,46,김서현,M,67,대전,2026-05-13
132,133,김성진,M,67,성남,2026-05-29
123,124,김시우,M,67,대구,2024-09-24
11,12,김정남,F,66,대전,2024-09-30
57,58,김성훈,F,66,성남,2025-06-14


주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

### 6. 주문 상세 금액 만들기

`line_total`은 주문 상세 1행의 금액입니다. 주문 상태를 연결하기 전 합계는 확정 매출이 아니라 전체 주문 상세 금액입니다.


In [21]:
order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

all_order_amount = order_items['line_total'].sum()
print('전체 주문 상세 금액:', all_order_amount)
display(order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())
print('아직 completed 주문만 필터링 된 상태는 아님')


전체 주문 상세 금액: 255610000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


아직 completed 주문만 필터링 된 상태는 아님


## 2. 필터링·정렬·파생 컬럼
- 적용한 필터 조건: age >= 30, city == '서울', city.isin(['서울', '부산'])
- 정렬 기준:  
    - products: price 내림차순 (ascending=False)
    - customers: age 내림차순 (ascending=False)
- 만든 파생 컬럼:`line_total`
- `line_total` 계산식: `line_total`==order_items['quantity'] * order_items['unit_price']`

![필터와 파생 컬럼](images/step02_transform.png)

### 결과 관찰
order_items에서 quantity*unit_price로곱해 line_total 파생 컬럼을 생성했다. 전체 주문 상세 금액의 총합은 255,610,000원으로 확인 되었다. Products 데이터를 price 기준 내림차순으로 정렬하여 가장 가격이 높은 상품 10개를 조회했다. 현재 금액 계산은 취소, 환불, 완료를 모두 포함한 전체 주문 대상으로, 완료 상태 주문은 따로 필터링하지 않은 상태이다.

### 나의 해석과 판단
30세 이상, 서울에 살거나 서울 혹은 부산(도시)에 사는 사람들로 필터링 하여 결과를 확인하였다. 
다만 `line_total`을 계산할 때는 그러한 필터링 없이 계산하였다. 따라서 completed 상태가 아닌 취소, 반품 상태의 주문까지 모두 포함해 계산한 상황이다. 
앞서 목표로 했던 것처럼 completed된 주문만 계산하면 현재의 line_total 값보다 작게 나타날 가능성이 높다. 현재는 취소되고 반품된 것까지 포함했으므로 실제 매출이 아니다. 따라서 전체 매출 금액이 과대 계산된 상황이라고 볼 수 있다. 실제로 유효한 매출만 집계하려면 order_status가 completed인 조건을 필터링하고 계산해야 한다. 이렇게 계산하는 경우 line_total 값이 작아질 것이다. 

### 업무·분석적 의미
line_total 파생 컬럼을 생성함으로써 단순 수량 분석이 아니라 실제 상품이나 주문 단위 등에서 실질적으로 매출에 어떤 영향을 미치는지 파악할 수 있다. 또한 정렬을 통해서 분포를 대략적으로 확인할 수 있다.
그리고 우선 필터링을 진행하면서 데이터의 분포를 대략적으로 한 번 더 확인해야 한다. 그리고 어떤 값을 기준으로 필터링할지도 이를 통해 명확하게 결정해야 한다. 이번에 진행한 것처럼 필터링 없이 계산하는 경우 실제로 분석하고자 헀던 값 외에 다른 값들이 끼어들어가 결과 분석이 혼선이 생길 수 있다.

### 한계와 추가 확인 사항
현재 결과는 주문 상태 필터링을 적용하지 않은 것으로, 순 매출이 아니라 총주문액을 계산한 결과이다. 따라서 추가로 orders 테이블과 order_items 테이블을 합쳐 completed 상태인 주문에 대해서만 line_total 합계를 다시 계산해야 한다. 또한 이때 이상치가 있지는 않은지 한번 더 확인해야 한다. unit_price가 이상한 값이 있거나, quantity 가 0 인 경우 등이 있는지 확인해야 한다.

### 7. 주문 데이터 병합과 검증

`validate='many_to_one'`은 주문 상세의 동일 주문 ID는 여러 번 나올 수 있지만 주문 테이블의 주문 ID는 한 번만 나와야 한다는 뜻입니다.


In [7]:
print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())

order_sales = order_sales.drop(columns='_merge')


orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

## 3. merge 검증
- 병합한 데이터:`order_items` 과 `orders`
- 사용한 key:`order_id`
- `validate` 결과: 우측 orders 테이블의 order_id 중복 수 = 0 로 many_to_one 검증 통과
- `indicator` 결과: both 764건, left_only 0건, right_only 0건
- 병합 전/후 행 수: 변화 없음 (병합 전 764행 -> 병합 후 764행)

![merge 검증](images/step03_merge.png)

### 결과 관찰
orders['order_id'].duplicated().sum() 결과가 0으로 나와, orders의 기본키 중복이 없음을 확인했다.
병합 전 후 행 수가 동일했으며, _merge 컬럼 확인 결과 모든 데이터가 both 상태로 확인되어 left_only 없이 모든 order_id가 orders와 연결되었다.

### 나의 해석과 판단
orders 테이블 내 order_id 중복이 0건이므로 행 수가 불어나는 등의 현상이 나타나지 않았고, indicator 결과 left_only가 0건이고 병합 전후 수가 동일하게 유지되었으므로 누락된 주문 내역 역시 없으므로 병합이 안전하다고 판단헀다.

### 업무·분석적 의미
잘못된 merge가 분석 결과에 어떤 영향을 줄 수 있는지 작성하세요.
merge가 잘못되어 중복으로 결합되는 등의 현상이 발생하면 동일한 주문 상품인데 중복으로 계산되어 전체 매출액이 실제보다 크게 계산되는 등의 오류가 발생할 수 있다. 혹은 키 값이 일치하지 않아서 데이터가 누락되는 등의 문제가 발생하면 실제 매출보다 적게 계산될 수 있으므로 역시 결과 해석에 혼선이 생긴다. 이 경우 과소평가하는 문제가 발생할 수 있다. 따라서 중복 체크를 하고, validate와 indciator 를 통해 merge가 제대로 되었는지 검증해야 데이터 분석에서 신뢰성을 확보할 수 있을 것이다.

### 한계와 추가 확인 사항
이번 병합은 order_items를 기준으로 했는데, 만약 orders에만 존재하고 order_items에는 없는 주문을 확인하려면 반대 방향(how=right)로 확인해보는 절차도 진행해야 할 것이다. 예를 들어 상품을 담지 않은 빈 주문 등이 있는 경우다. 
또한 키 관계가 정상이라고 해도 이 자체만으로 각 주문 값이 업무적으로 정확한 것이라고 보장하기에는 어렵다.

### 8. 완료 주문만 선택하기

이후의 매출 요약은 `order_status == 'completed'`인 주문만 사용합니다.


In [ ]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
print('날짜 변환 실패:', order_sales['order_date'].isna().sum())

completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

completed_order_sales['order_month'] = (
    completed_order_sales['order_date'].dt.to_period('M').astype(str)
)

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())


날짜 변환 실패: 0
전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


### 9. 상품 데이터 병합과 검증


In [9]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())

completed_sales_items = completed_sales_items.drop(columns='_merge')


products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

### 10. 카테고리별·상품별 매출


In [27]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)
category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)
display(category_sales)
product_sales = (
    completed_sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)
display(product_sales.head(10))

,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


### 11. 월별 매출


In [11]:
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_summary['average_order_value'] = (
    monthly_summary['total_sales'] / monthly_summary['order_count']
).round(0)

display(monthly_summary)


,order_month,total_sales,order_count,average_order_value
0,2025-07,5869000,8,733625.0
1,2025-08,15621000,18,867833.0
2,2025-09,10190000,13,783846.0
3,2025-10,25766000,26,991000.0
4,2025-11,8812000,12,734333.0
5,2025-12,11501000,14,821500.0
6,2026-01,17423000,22,791955.0
7,2026-02,9749000,17,573471.0
8,2026-03,13429000,14,959214.0
9,2026-04,17553000,23,763174.0


In [23]:
print(completed_order_sales["order_date"].min())
print(completed_order_sales["order_date"].max())
print('첫번째달, 마지막달의 데이터가 full로 되어 있지 않으면 월별 매출 분석 시 감안해야 함')

2025-07-09 00:00:00
2026-07-05 00:00:00
첫번째달, 마지막달의 데이터가 full로 되어 있지 않으면 월별 매출 분석 시 감안해야 함


### 12. 고객별 구매 금액

고객 ID로 먼저 집계한 뒤 고객 속성을 붙입니다. 출력에는 실제 이름 대신 익명화된 고객 라벨을 사용합니다.


In [12]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


## 4. completed 주문 범위와 집계
- 분석 범위 정의: `order_status == "completed"`, 분석 날짜 범위: 2025-07-09 ~ 2026-07-05
- 카테고리별 결과: 스포츠 카테고리가 매출액 31743000원(21.31%)으로 1위, 패션 카테고리가 10587000원(7.11%)으로 최하위였다. 
- 상품별 결과: '스포츠 상품 041'이 매출액 5705000원(수량 35개)으로 상품 중 1위였다.
- 월별 결과: 2025년 10월 매출이 25766000원(26건)으로 가장 높았고, 2026년 7월이 2188000원(2건)으로 가장 낮았다.
- 고객별 결과: 1위는 성남의 Customer 117로 주분 5건, 금액 4100000원이었다.

![핵심 집계 결과](images/step04_groupby.png)

### 결과 관찰
- 카테고리: 스포츠(21.31%), 전자기기(17.72%), 생활용품(16.05%) 3개 카테고리가 전체 매출의 절반 이상(약 55%)을 차지했다.
- 상품: '스포츠 상품 041'과 '식품 상품 012'가 각각 매출 상위 1, 2위를 기록했다. 상위 10개에 해당하는 품목의 카테고리는 스포츠, 식품, 뷰티, 전자기기, 패션, 생활용품이었다. 
- 월별 추이: 2025년 10월과 2026년 1월, 4월에 매출 피크(수량)가 발생했다. 평균 주문 금액의 경우 2025년 10월, 2025년 12월, 2026년 3월, 2026년 7월에 피크였다. 반면 범위 첫 달인 2025년 7월(8건)과 마지막 달인 2026년 7월(2건)은 주문 건수와 매출이 상대적으로 낮게 집계되었다.
- 고객: 상위 구매 고객층은 성남, 고양, 수원, 서울 등 주로 수도권 지역 거주자가 다수를 차지했다.

### 나의 해석과 판단
카테고리별 결과가 가장 중요하다고 판단했다. 특히 스포츠가 31743000원과 21.31%로 가장 큰 completed 주문 금액을 가지고 있었다. 그러나 스포츠 카테고리의 금액이 큰 이유를 지금 판단하기는 어렵다. 매출과 수량 모두 많았으나, 상품 각각의 단가나 구성 등도 금액에 영향을 줄 수 있으므로 이를 분리해서 확인해야 한다.
또한 월별 분석 시 범위의 첫달과 마지막 달은 전체 30일을 포함하지 않아 상대적으로 주문 건수가 낮게 확인되었다. 또한 2026년 7월의 경우 범위가 모든 30일을 커버하지 않아 2건밖에 주문 수량이 없었음에도 불구하고 매출금액이 매우 높았다. 따라서 이러한 요소들을 주의해서 판단해야 한다. 특히 첫달과 마지막달을 단순히 낮게 매출이 나온 날이라고 판단해서는 안된다 .

### 업무·분석적 의미
앞서 분석한 결과와 같은 월별 패턴이 연마다 반복되어 나타나고, 주기적으로 매출이 상승하는 시기가 있는 경우 이 시기에 맞춘 마케팅을 진행할 수 있을 것이다. 또한 매출 비중이 높은 카테고리의 재고를 안정적으로 확보하고, 매출이 낮은 카테고리리는 상품을 재검토할 필요가 있다. 또한 월별 분석 시 모든 기간이 포함되었는지 제대로 확인해야 적절한 분석을 진행할 수 있다.

### 한계와 추가 확인 사항
total_sales는 completed 주문의 `quantity × unit_price` 합계이다. 이것에는 할인, 쿠폰, 배송비, 원가 등이 포함되지 않았으므로 이 결과를 회계상 순매출이나 영업이익과 같다고 단정할 수 없다. 또한 집계 기간의 첫 달과 마지막 달은 일부 날짜만 포함된 부분 기간이고, 카테고리별 매출만으로 실제 수요나 수익성을 판단하기에는 아직 성급하다.


In [26]:

base_total = completed_order_sales["line_total"].sum()
category_total = category_sales["total_sales"].sum()
product_total = product_sales["total_sales"].sum()
monthly_total = monthly_summary["total_sales"].sum()
customer_total = customer_sales["total_sales"].sum()

print("base:", base_total)
print("category:", category_total)
print("product:", product_total)
print("month:", monthly_total)
print("customer:", customer_total)

if base_total == category_total == product_total == monthly_total == customer_total:
    print("모두 총합 일치")
else:
    print("총합 불일치")



base: 148990000
category: 148990000
product: 148990000
month: 148990000
customer: 148990000
모두 총합 일치


## 5. 총합 일치 검증
- 원본 completed `line_total` 합계: 148990000
- 카테고리 합계: 148990000
- 월별 합계: 148990000
- 고객별 합계:148990000
- 차이 여부: 없다. (동일하)

![총합 검증](images/step05_total_check.png)

### 나의 해석과 판단
동일한 completed 범위를 카테고리, 상품, 월, 고객 단위로 각각 다시 집계해도 총합이 모두 148990000원으로 정확히 일치했다. 따라서 현재 범위 내에서는 필터링, 병합, groupby 과정에서 금액 중복이나 누락된 값이 없을 가능성이 높다고 판단할 수 있다. 이처럼 총합 검증이 필요한 이유는, 필터링, 병합, groupby 과정에서 중복이나 누락된 값이 없음을 다시 확인하기 위해서다. 이러한 일이 발생하는 경우 데이터 분석을 신뢰하기 어려우므로, 미리 점검하고 넘어가야 한다. 다만 총합이 일차한다고 해서 각 데이터 값 자체가 업무적으로도 정확하다고 판단하기에는 어렵다.
만약 총합이 불일치하는 상황이 발생하는 경우 
1. completed 필터가 동일한지, 
2. line_total 계산식이 동일한지, 
3. merge 후 행 수가 증가하거나 감소하지 않았는지,
4. groupby에서 결측 key가 빠지지 않았는가, 
5. 일부 데이터만 다시 필터링하지 않았는가, 
6. 숫자 타입이 모두 정상적인지
의 순서대로 확인해야 한다.

### 13. 결과 저장하기


In [13]:
outputs = {
    'ch04_category_sales.csv': category_sales,
    'ch04_product_sales.csv': product_sales,
    'ch04_monthly_sales.csv': monthly_summary,
    'ch04_customer_sales.csv': customer_sales,
}

for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(filename, path.exists(), path.stat().st_size)


ch04_category_sales.csv True 254
ch04_product_sales.csv True 4409
ch04_monthly_sales.csv True 442
ch04_customer_sales.csv True 3387


In [29]:
saved_category_sales = pd.read_csv(
    REPORT_DIR / "ch04_category_sales.csv"
)

print(saved_category_sales.shape)
print(saved_category_sales.columns.tolist())
saved_category_sales.head()

(7, 4)
['category', 'total_quantity', 'total_sales', 'sales_ratio']


,category,total_quantity,total_sales,sales_ratio
0,스포츠,295,31743000,21.31
1,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
3,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12


In [ ]:
#수정전 = 오류 남 
import pandas as pd

orders = pd.read_csv("orders.csv")
items = pd.read_csv("order_items.csv")

# 1. line_total
items["line_total"] = items["quantity"] * items["unit_price"]

# 2. orders.order_id 고유성 확인
assert orders["order_id"].is_unique, "orders.order_id is not unique"

# 3~4. many-to-one merge + indicator, 병합 전후 행 수/미매칭 확인
n_before = len(items)
df = items.merge(
    orders, on="order_id",
    how="left", validate="many_to_one", indicator=True
)
print("rows before/after:", n_before, len(df))
print("unmatched:", (df["_merge"] != "both").sum())

# 5. 날짜 변환 실패 건수
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
print("date conversion failures:", df["order_date"].isna().sum())

# 6. completed만 필터링
completed = df[df["order_status"].eq("completed")].copy()

# 7. 월별 total_sales + 고유 order_count
completed["month"] = completed["order_date"].dt.to_period("M")
monthly = completed.groupby("month").agg(
    total_sales=("line_total", "sum"),
    order_count=("order_id", "nunique")
).reset_index()

# 8. 원본 completed total과 월별 합계 비교
completed_total = completed["line_total"].sum()
monthly_total = monthly["total_sales"].sum()

print(monthly)
print("completed total:", completed_total)
print("monthly total:", monthly_total)
print("difference:", completed_total - monthly_total)

FileNotFoundError: [Errno 2] No such file or directory: 'orders.csv'

In [31]:
#수정후

#수정전 = 오류 남 
import pandas as pd

orders = pd.read_csv(DATA_DIR / 'orders.csv')
items = pd.read_csv(DATA_DIR / 'order_items.csv')

# 1. line_total
items["line_total"] = items["quantity"] * items["unit_price"]

# 2. orders.order_id 고유성 확인
assert orders["order_id"].is_unique, "orders.order_id is not unique"

# 3~4. many-to-one merge + indicator, 병합 전후 행 수/미매칭 확인
n_before = len(items)
df = items.merge(
    orders, on="order_id",
    how="left", validate="many_to_one", indicator=True
)
print("rows before/after:", n_before, len(df))
print("unmatched:", (df["_merge"] != "both").sum())

# 5. 날짜 변환 실패 건수
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
print("date conversion failures:", df["order_date"].isna().sum())

# 6. completed만 필터링
completed = df[df["order_status"].eq("completed")].copy()

# 7. 월별 total_sales + 고유 order_count
completed["month"] = completed["order_date"].dt.to_period("M")
monthly = completed.groupby("month").agg(
    total_sales=("line_total", "sum"),
    order_count=("order_id", "nunique")
).reset_index()

# 8. 원본 completed total과 월별 합계 비교
completed_total = completed["line_total"].sum()
monthly_total = monthly["total_sales"].sum()

print(monthly)
print("completed total:", completed_total)
print("monthly total:", monthly_total)
print("difference:", completed_total - monthly_total)

rows before/after: 764 764
unmatched: 0
date conversion failures: 0
      month  total_sales  order_count
0   2025-07      5869000            8
1   2025-08     15621000           18
2   2025-09     10190000           13
3   2025-10     25766000           26
4   2025-11      8812000           12
5   2025-12     11501000           14
6   2026-01     17423000           22
7   2026-02      9749000           17
8   2026-03     13429000           14
9   2026-04     17553000           23
10  2026-05      8063000           11
11  2026-06      2826000            4
12  2026-07      2188000            2
completed total: 148990000
monthly total: 148990000
difference: 0



## 6. LLM pandas 코드 검증
- LLM Prompt 요약: orders와 order_items (CSV 파일 이름임)를 사용해 completed 주문 기준 월별 금액을 계산하려고 한다. 조건은 (코드는 짧게, line_total 생성할 것, id 고유성 확인할 것, merge 및 indicator 사용할 것, 병합전후 행수,미매칭 확인할 것, 날짜변환실패 확인할 것, completed 기준 필터링 및 월별 total_sales와 고유 order count 계산할 것, 원본 total과 월별 합꼐 비교할 것, 결과를 회계상 순매출이라 단정하지 말것)
- 제안 코드 요약:
    - `line_total` 계산 및 `orders.order_id` 고유성(`assert`) 검증
    - `many_to_one` 방식의 Left Merge 및 미매칭/행 수 변화 검증
    - `pd.to_datetime`을 통한 날짜 변환 및 결측치 확인
    - `completed` 상태 필터링 후 월별(`dt.to_period('M')`) 총 매출액 및 고유 주문 수(`nunique`) 집계
    - 원본 완료 주문 총액과 월별 집계 총액 간의 차이(`difference`) 검증
- 실제 컬럼/범위와 맞지 않은 부분:
    - 파일 경로가 단순 파일명(`orders.csv`,`order_itmes.csv`)으로만 되어 있어 프로젝트 경로 구조(`DATA_DIR / 파일`)와 달라 오류가 났다.
- 수정한 내용: 다음 두 코드로 수정하였다.
    - orders = pd.read_csv(DATA_DIR / 'orders.csv')
    - items = pd.read_csv(DATA_DIR / 'order_items.csv')

- 최종 판단: 수정 후 사용 

![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단
에러 없이 실행된다고 해서 항상 분석 결과를 적절히 나타냈다고 보기에는 어렵다. 문법적으로는 에러가 없어도, 분석하는 방법이나 그 결과가 내 목적에 맞다고 보장하기 어렵다. 예를 들어 LLM이 데이터의 내부 의미 (ex. completed 상태 외 다른 상태 존재 여부 - 이번 분석에서는 내가 지정해주었지만, 다른 내가 생각하지 못한 요소가 있을 수 있음)를 완벽하게 이해하지 못하고 입력된 텍스트 조건만 기계적으로 적용할 수 있다. 또한 수집 범위의 시작/종료일 등으로 월이 완벽하게 반영되지 않았는데, 이러한 점을 놓치고 월별 패턴을 분석하는 등 데이터가 갖는 다른 요소를 스스로 판단하지 않을 수 있다. 따라서 LLM이 작성한 코드는 사전 검증을 거치고, 내 자신의 비판적인 해석 역시 더해서 활용해야 한다.

### 14. LLM 코드 검증 연습

```text
LLM이 다음 코드를 제안했습니다.

category_sales = order_items.groupby('category')['line_total'].sum()

현재 데이터에서 이 코드가 바로 실행 가능한지 검토해 주세요.
category 컬럼의 위치, 주문 상태 필터링, merge validate와 indicator를 포함해
안전한 수정 코드와 검증 순서를 설명해 주세요.
```


In [34]:
category_sales = order_items.groupby('category')['line_total'].sum()

KeyError: 'category'

In [35]:
# 질문 6 추가 검증: LLM 제안 코드
print("order_items에 category 존재:", 'category' in order_items.columns)
print("order_items 컬럼:", order_items.columns.tolist())


all_scope_items = order_items.merge(
    products[['product_id', 'category']],
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)
all_scope_total = all_scope_items['line_total'].sum()
corrected_total = category_sales['total_sales'].sum()

print('orders.order_id 중복 수:', int(orders['order_id'].duplicated().sum()))
print('products.product_id 중복 수:', int(products['product_id'].duplicated().sum()))
print('전체 상태 상품 연결 미매칭:', int((all_scope_items['_merge'] == 'left_only').sum()))
print('주문 상태 실제 값:', orders['order_status'].value_counts().to_dict())
print('상태 미필터 전체 금액:', int(all_scope_total))
print('수정 후 completed 카테고리 합계:', int(corrected_total))
print('범위 차이 금액:', int(all_scope_total - corrected_total))
print('수정 후 카테고리 수:', len(category_sales))


order_items에 category 존재: False
order_items 컬럼: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']
orders.order_id 중복 수: 0
products.product_id 중복 수: 0
전체 상태 상품 연결 미매칭: 0
주문 상태 실제 값: {'completed': 184, 'cancelled': 64, 'refunded': 52}
상태 미필터 전체 금액: 255610000
수정 후 completed 카테고리 합계: 148990000
범위 차이 금액: 106620000
수정 후 카테고리 수: 7


해당 코드는 바로 실행하는 것이 불가능하다.
`category_sales = order_items.groupby('category')['line_total'].sum()`
실제 `order_items` 컬럼에는 `category`가 없기 때문에 `KeyError: 'category'`로 실패했다. `category`는 `products`에 있으므로 `order_items`의 `product_id`와 `products`의 `product_id`를 merge하여 연결해야 한다. 또한 주문상태가 `orders`에 있기 때문에 `order_id` merge 후 `completed` 필터링을 해야 한다. 또한 데이터에는 `line_total`이 없으므로 이 역시 내가 quantity 와 unit_price를 곱해 생성해야 한다.

검증 순서는 다음과 같다. (수정코드는 아래에)
1) order_items에서 수량과 단가를 곱해 line_total 컬럼 생성
2) orders 테이블의 order_id 중복 여부를 체크하여 고유성 확인
3) order_id 기준으로 orders 테이블을 병합하며 validate="many_to_one" 옵션을 적용
4) 병합 전후의 전체 행 수와 _merge 컬럼 결과를 확인해 행 증가나 미매칭이 없는지 검증
5) order_status == "completed" 조건인 유효 주문만 필터링
6) products 테이블의 product_id 고유성을 확인
7) product_id 기준으로 products 테이블을 병합하고, 동일하게 validate, indicator, 행 수 변화를 검증
8) 카테고리별로 line_total 합계를 계산
9) 카테고리별 합계의 총합이 원본 completed 주문의 전체 금액 합계와 일치하는지 확인

In [36]:
#수정코드 및 설명
# 1. 주문 상세 금액 생성
order_items_work = order_items.copy()
order_items_work["line_total"] = (
    order_items_work["quantity"]
    * order_items_work["unit_price"]
)

# 2. orders 키 고유성 확인
print(
    "orders.order_id 중복 수:",
    orders["order_id"].duplicated().sum(),
)

# 3. orders 병합: 주문 상태 연결
order_sales = order_items_work.merge(
    orders[[
        "order_id",
        "order_status",
        "order_date",
    ]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("orders 병합 전 행 수:", len(order_items_work))
print("orders 병합 후 행 수:", len(order_sales))
print(order_sales["_merge"].value_counts(dropna=False))

order_sales = order_sales.drop(columns="_merge")

# 4. completed 주문만 선택
completed_order_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()

print(
    completed_order_sales["order_status"]
    .value_counts(dropna=False)
)

# 5. products 키 고유성 확인
print(
    "products.product_id 중복 수:",
    products["product_id"].duplicated().sum(),
)

# 6. products 병합: category 연결
completed_sales_items = completed_order_sales.merge(
    products[[
        "product_id",
        "category",
    ]],
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("products 병합 전 행 수:", len(completed_order_sales))
print("products 병합 후 행 수:", len(completed_sales_items))
print(
    completed_sales_items["_merge"]
    .value_counts(dropna=False)
)

completed_sales_items = completed_sales_items.drop(columns="_merge")

# 7. 카테고리별 completed 주문 금액 집계
category_sales = (
    completed_sales_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
    )
    .sort_values("total_sales", ascending=False)
)

display(category_sales)

# 8. 원본 completed 금액과 카테고리 합계 비교
base_total = completed_order_sales["line_total"].sum()
category_total = category_sales["total_sales"].sum()

print("원본 completed 금액:", base_total)
print("카테고리별 합계:", category_total)
print("총합 일치:", base_total == category_total)

orders.order_id 중복 수: 0
orders 병합 전 행 수: 764
orders 병합 후 행 수: 764
_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64
order_status
completed    474
Name: count, dtype: int64
products.product_id 중복 수: 0
products 병합 전 행 수: 474
products 병합 후 행 수: 474
_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64


,category,total_sales
3,스포츠,31743000
5,전자기기,26400000
2,생활용품,23915000
1,뷰티,23383000
4,식품,16573000
0,도서,16389000
6,패션,10587000


원본 completed 금액: 148990000
카테고리별 합계: 148990000
총합 일치: True


### 15. 실습 과제

1. 40세 이상 고객을 추출합니다.
2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
3. 결제수단별 주문 수와 비율을 계산합니다.
4. 카테고리별 평균 상품 가격을 계산합니다.
5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.


In [37]:
# 과제 코드를 아래에 작성하세요.
# 1. 40세 이상 고객 추출
customers_over_40 = customers[
    customers["age"] >= 40
]
display(customers_over_40.head())
print("40세 이상 고객 수:", len(customers_over_40))

# 2. 상품 가격이 낮은 순서대로 10개 출력
lowest_price_products = (
    products
    .sort_values("price", ascending=True)
    .head(10)
)
display(lowest_price_products)

# 3. 결제수단별 주문 수와 비율 계산
payment_summary = (
    orders["payment_method"]
    .value_counts(dropna=False)
    .rename_axis("payment_method")
    .reset_index(name="order_count")
)

payment_summary["order_ratio"] = (
    payment_summary["order_count"]
    / payment_summary["order_count"].sum()
    * 100
).round(2)

display(payment_summary)

# 4. 카테고리별 평균 상품 가격 계산
category_average_price = (
    products
    .groupby("category", as_index=False)
    .agg(
        average_price=("price", "mean"),
    )
    .sort_values("average_price", ascending=False)
)

category_average_price["average_price"] = (
    category_average_price["average_price"]
    .round(0)
)

display(category_average_price)


# 5. 완료 주문과 전체 주문의 금액 차이 계산
all_order_total = order_items["line_total"].sum()

completed_order_total = (
    completed_order_sales["line_total"]
    .sum()
)

excluded_order_total = (
    all_order_total
    - completed_order_total
)

print("전체 주문 상세 금액:", all_order_total)
print("completed 주문 금액:", completed_order_total)
print("전체 - completed 차이:", excluded_order_total)

,customer_id,name,gender,age,city,signup_date
2,3,이경수,F,61,성남,2024-06-12
3,4,조영호,F,55,울산,2026-04-13
6,7,이상현,F,53,인천,2024-12-12
8,9,송지민,M,69,서울,2025-10-19
9,10,유도현,F,62,울산,2024-10-06


40세 이상 고객 수: 78


,product_id,product_name,category,price
5,6,전자기기 상품 006,전자기기,5000
27,28,뷰티 상품 028,뷰티,5000
97,98,스포츠 상품 098,스포츠,10000
46,47,생활용품 상품 047,생활용품,11000
94,95,전자기기 상품 095,전자기기,20000
49,50,뷰티 상품 050,뷰티,23000
82,83,전자기기 상품 083,전자기기,24000
86,87,도서 상품 087,도서,25000
41,42,패션 상품 042,패션,28000
58,59,뷰티 상품 059,뷰티,32000


,payment_method,order_count,order_ratio
0,kakao_pay,79,26.33
1,naver_pay,77,25.67
2,bank_transfer,74,24.67
3,card,70,23.33


,category,average_price
4,식품,137143.0
1,뷰티,117688.0
6,패션,115909.0
3,스포츠,111579.0
0,도서,106857.0
5,전자기기,101588.0
2,생활용품,96438.0


전체 주문 상세 금액: 255610000
completed 주문 금액: 148990000
전체 - completed 차이: 106620000


### 6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트
orders, order_items, customers 데이터를 사용해 completed 주문 기준 고객별 구매 금액을 계산하는 pandas 코드를 작성해줘. 다음 조건을 지켜.

1. quantity × unit_price로 line_total을 생성할 것
2. orders.order_id의 고유성을 확인할 것
3. order_items와 orders를 order_id로 left merge할 것
4. merge에 validate="many_to_one"과 indicator=True를 사용할 것
5. 병합 전후 행 수와 미매칭 여부를 확인할 것
6. order_status == "completed"만 필터링할 것
7. customer_id별 고유 주문 수와 total_sales를 계산할 것
8. customers의 customer_id 고유성을 확인할 것
9. 고객 속성은 city만 연결할 것
10. 고객 병합에도 적절한 validate를 사용할 것
11. 원본 completed 금액과 고객별 total_sales 합계가 일치하는지 검증할 것
12. 결과를 회계상 순매출이나 이익이라고 단정하지 말 것

### 정리

이번 장에서는 실제 값 확인, 컬럼 선택, 필터링, 정렬, 파생 컬럼, 병합 검증, 완료 주문 기준 집계, CSV 저장 과정을 수행했습니다. 다음 장에서는 결측치, 중복, 타입 오류, 날짜 형식, 이상값 후보를 다루는 데이터 전처리로 이어집니다.


## 7. Chapter 04 최종 인사이트
### 가장 의미 있다고 생각한 결과 2가지
1. completed 주문 기준 스포츠 카테고리의 total_sales가 31743000원으로 가장 컸다(전체의 21.31%).
2. 월별 total_sales는 2025년 10월이 25766000원으로 가장 컸다 (26건, 평균 주문 금액 991000원)

### 그 결과를 뒷받침하는 수치/표
![핵심 집계 결과](images/step07_1.png)
![핵심 집계 결과](images/step07_2.png)

### 추가로 확인하고 싶은 질문
스포츠 카테고리의 높은 매출 원인은 무엇인가?  (ex. 판매 수량, 평균 단가, 특정 상품 집중) 

### 현재 결과의 한계
현재 분석 범위가 2025년 7월 9일~2026년 7월 5일이기 때문에 첫달/마지막달은 완전한 30일의 데이터가 아니므로 각 월의 집계액을 온전히 분석하기에 어렵다. 또한 total_sales에 세금이나 할인, 배송비, 원가 등이 포함되지 않아 회계상 순매출이나 이익이라고 보기 어렵다. 따라서 현재 분석 결과 만으로는 유ㅜ의미한 매출에 대한 인과관계 등 분석이 어렵다.

## 최종 제출 체크
- [X] 핵심 셀 Output이 남아 있습니다.
- [X] merge와 총합 검증 Evidence가 있습니다.
- [X] 결과 관찰과 해석이 구분되어 있습니다.
- [X] LLM 코드를 검증했습니다.
- [X] 개인정보/Secret이 없습니다.
- [X] `chapter04/chapter04.ipynb`가 GitHub에서 정상 표시됩니다.
- [X] 최종 Notebook 파일 URL을 제출합니다.